In [18]:
import requests
import json
from datetime import datetime
from pprint import pprint

OLLAMA_URL = "http://localhost:11434/api/chat"

def query_ollama(prompt, system_prompt=None, model="mistral"):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    
    payload = {
        "model": model,
        "messages": messages,
        "stream": False
    }

    response = requests.post(OLLAMA_URL, json=payload)
    response.raise_for_status()
    return response.json()["message"]["content"]


In [19]:
def generate_sketch(task_description):
    system_prompt = (
        "You are a symbolic reasoning engine. You output structured proof sketches in JSON format.\n"
        "Use methods like induction, contradiction, symmetry, or analogical reasoning.\n"
        "Be precise, structured, and minimal. Output JSON only."
    )

    prompt = f"""TASK: {task_description}

Respond in JSON with:
- problem: str
- assumptions: [str]
- methods: [str]
- symbolic_steps: [{{step: str, reasoning: str}}]
- conclusion: str

Only output valid JSON.
"""
    raw_output = query_ollama(prompt, system_prompt)
    
    # Extract first valid JSON object from streaming-like text
    start_idx = raw_output.find('{')
    end_idx = raw_output.rfind('}')
    
    try:
        json_obj = json.loads(raw_output[start_idx:end_idx+1])
        return json_obj
    except json.JSONDecodeError as e:
        print("Failed to parse JSON:")
        print(raw_output)
        raise e


In [20]:
def evaluate_sketch(sketch):
    evaluations = []
    for i, step in enumerate(sketch.get("symbolic_steps", [])):
        step_text = step["step"]
        reasoning_text = step["reasoning"]
        
        # Basic syntactic validation (can be expanded later)
        status = "ok" if isinstance(step_text, str) and isinstance(reasoning_text, str) else "fail"
        comment = "Step and reasoning syntactically coherent." if status == "ok" else "Invalid format."

        evaluations.append({
            "step_index": i,
            "step": step_text,
            "reasoning": reasoning_text,
            "evaluation": {
                "status": status,
                "comments": comment
            }
        })
    
    return evaluations


In [21]:
task = "Propose a unified theory of physics"

sketch = generate_sketch(task)
evaluations = evaluate_sketch(sketch)

output = {
    "proof_sketch": sketch,
    "evaluations": evaluations
}

pprint(output)


{'evaluations': [{'evaluation': {'comments': 'Step and reasoning syntactically '
                                             'coherent.',
                                 'status': 'ok'},
                  'reasoning': 'Induction: These laws have been extensively '
                               'tested and hold true in a wide range of '
                               'physical scenarios.',
                  'step': '1. Combine the laws of classical mechanics, '
                          'electromagnetism, and special relativity',
                  'step_index': 0},
                 {'evaluation': {'comments': 'Step and reasoning syntactically '
                                             'coherent.',
                                 'status': 'ok'},
                  'reasoning': 'Analogical reasoning: Quantum mechanics '
                               'describes the behavior of particles at very '
                               'small scales, which cannot be explained by '
        

In [22]:
def save_output(sketch, evaluations):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_name = f"hsde_output_unified_physics_{timestamp}"

    # JSON
    with open(f"{base_name}.json", "w") as f:
        json.dump({"proof_sketch": sketch, "evaluations": evaluations}, f, indent=2)

    # Markdown
    with open(f"{base_name}.md", "w") as f:
        f.write(f"# HSDE Task: {sketch['problem']}\n\n")
        f.write("## Assumptions\n")
        for a in sketch["assumptions"]:
            f.write(f"- {a}\n")
        f.write("\n## Methods\n")
        for m in sketch["methods"]:
            f.write(f"- {m}\n")
        f.write("\n## Symbolic Reasoning Steps\n")
        for i, step in enumerate(sketch["symbolic_steps"]):
            eval_comment = evaluations[i]["evaluation"]["comments"]
            f.write(f"**Step {i+1}:** {step['step']}\n")
            f.write(f"- Reasoning: {step['reasoning']}\n")
            f.write(f"- Evaluation: {eval_comment}\n\n")
        f.write("## Conclusion\n")
        f.write(f"{sketch['conclusion']}\n")

save_output(sketch, evaluations)
